# Feature-Based vs. Deep Learning Models for 12-Lead ECG Classification
**BME 304L Final Project — Sara Yoon (syy3), Jackie No (jjn22)**

This notebook runs the full analysis described in the Methods, in order:

| Section | What it does | Output |
|---|---|---|
| 0 | Setup | – |
| 1 | Download PTB-XL and PTB-XL+ into `data/` (first run only) | – |
| 2 | Build labels and Table 1 | `results/table1_counts.csv` |
| 3 | Load, filter, and standardize the 100 Hz signals | – |
| 4 | Prepare the PTB-XL+ (12SL) features | – |
| 5 | Tune and train the random forests | `results/rf_*.csv/json` |
| 6 | Train the CNN (3 seeds) | `figures/fig2_training_curves.png` |
| 7 | **Stop:** fix the expected lead groups and commit them | `results/expected_lead_groups.json` |
| 8 | RQ1: test-set AUROC, bootstrap CIs, paired comparison | Table 2, Fig. 3 |
| 9 | RQ2: lead-group SHAP (forest) and occlusion (CNN) | Table 3, Fig. 4, Fig. 5 |
| 10 | Summary of every number needed for the paper | `results/results_summary.json` |

**How to run:** select the `.venv` kernel, then run cells top to bottom. The first full run takes roughly 1–2 hours (download about 10 min, forest tuning 20–60 min, CNN about 20–40 min). Slow steps save their results, so rerunning the notebook later skips them.

**Fold 10 (the test set) is only used from Section 8 on.**

## 0. Setup

In [ ]:
import ast, itertools, json, platform, sys, time, urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import scipy
from scipy.signal import butter, filtfilt
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import wfdb
import shap
from tqdm.auto import tqdm

# Project folders (works whether this notebook is in the repo root or in notebooks/)
ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent
DATA, RESULTS, FIGS, MODELS = (ROOT / d for d in ["data", "results", "figures", "models"])
for d in (DATA, RESULTS, FIGS, MODELS):
    d.mkdir(exist_ok=True)

SEED = 42                 # main seed (forest, bootstrap, primary CNN)
CNN_SEEDS = [42, 43, 44]  # CNN repeated with 3 seeds for stability
N_BOOT = 2000             # bootstrap resamples
CLASSES = ["NORM", "MI", "CD"]
LEADS = ["I", "II", "III", "AVR", "AVL", "AVF", "V1", "V2", "V3", "V4", "V5", "V6"]  # PTB-XL order
GROUPS = {"Inferior": ["II", "III", "AVF"],
          "Anterior": ["V1", "V2", "V3", "V4"],
          "Lateral": ["I", "AVL", "V5", "V6"]}          # aVR is in no group
BBB = {"CRBBB", "IRBBB", "CLBBB", "ILBBB"}             # bundle branch block subclasses

if torch.backends.mps.is_available():
    device = torch.device("mps")      # Apple-chip GPU
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
print("Project folder:", ROOT)
print("Device:", device)

## 1. Download the data (first run only)

Downloads the PTB-XL v1.0.3 zip (about 1.7 GB) into `data/`, extracts only the 100 Hz signals and the CSV files, and downloads the two PTB-XL+ files we need. `data/` is listed in `.gitignore`, so none of this goes into git.

If the download fails (for example on a slow network), download the zip manually from https://physionet.org/content/ptb-xl/1.0.3/ ("Download the ZIP file"), save it as `data/ptb-xl-1.0.3.zip`, and rerun this cell.

In [ ]:
PTBXL_ZIP = DATA / "ptb-xl-1.0.3.zip"
PTBXL_URLS = [
    "https://physionet.org/static/published-projects/ptb-xl/"
    "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3.zip",
    "https://physionet.org/content/ptb-xl/get-zip/1.0.3/",
]
PLUS_DIR = DATA / "ptbxl_plus"
PLUS_URL = "https://physionet.org/files/ptb-xl-plus/1.0.1/features/"
PLUS_FILES = ["12sl_features.csv", "feature_description.csv"]


def download(url, dest):
    tmp = dest.with_name(dest.name + ".part")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
        total = int(r.headers.get("Content-Length", 0)) or None
        with tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))
    tmp.rename(dest)


def find_ptbxl_dir():
    hits = sorted(DATA.rglob("ptbxl_database.csv"))
    return hits[0].parent if hits else None


PTBXL_DIR = find_ptbxl_dir()
if PTBXL_DIR is None:
    if not PTBXL_ZIP.exists():
        for url in PTBXL_URLS:
            try:
                download(url, PTBXL_ZIP)
                break
            except Exception as e:
                print("Download failed from", url, "->", e)
        else:
            raise RuntimeError("Download the zip manually (see note above) and save it as data/ptb-xl-1.0.3.zip")
    print("Extracting the 100 Hz signals and CSV files (the 500 Hz folder is skipped)...")
    with zipfile.ZipFile(PTBXL_ZIP) as z:
        members = [m for m in z.namelist() if "/records500/" not in m]
        z.extractall(DATA / "ptbxl", members=members)
    PTBXL_DIR = find_ptbxl_dir()

PLUS_DIR.mkdir(exist_ok=True)
for name in PLUS_FILES:
    if not (PLUS_DIR / name).exists():
        download(PLUS_URL + name, PLUS_DIR / name)

print("PTB-XL folder:", PTBXL_DIR)
print("PTB-XL+ files:", sorted(p.name for p in PLUS_DIR.iterdir()))
print("Once this works, you can delete data/ptb-xl-1.0.3.zip to free 1.7 GB.")

## 2. Labels, splits, and Table 1 (Methods 2.1)
Diagnostic statements are mapped to superclasses and subclasses with `scp_statements.csv`. ECGs with no diagnostic superclass are excluded. STTC-only and HYP-only ECGs are kept as negatives for all three classes.

In [ ]:
db = pd.read_csv(PTBXL_DIR / "ptbxl_database.csv", index_col="ecg_id")
scp = pd.read_csv(PTBXL_DIR / "scp_statements.csv", index_col=0)
db["scp_codes"] = db["scp_codes"].apply(ast.literal_eval)
diag = scp[scp["diagnostic"] == 1]


def collect(codes, column):
    return {diag.loc[c, column] for c in codes if c in diag.index}


db["superclasses"] = db["scp_codes"].apply(lambda c: collect(c, "diagnostic_class"))
db["subclasses"] = db["scp_codes"].apply(lambda c: collect(c, "diagnostic_subclass"))

n_before = len(db)
db = db[db["superclasses"].apply(len) > 0].sort_index()
print(f"Excluded {n_before - len(db)} ECGs with no diagnostic superclass; {len(db):,} remain.")

Y = np.stack([db["superclasses"].apply(lambda s, c=c: c in s).to_numpy() for c in CLASSES],
             axis=1).astype(np.float32)
fold = db["strat_fold"].to_numpy()
patient = db["patient_id"].to_numpy()
tr, va, te = np.where(fold <= 8)[0], np.where(fold == 9)[0], np.where(fold == 10)[0]


def has_sub(name):
    return db["subclasses"].apply(lambda s: name in s).to_numpy()


is_imi, is_ami = has_sub("IMI"), has_sub("AMI")
is_bbb = db["subclasses"].apply(lambda s: len(s & BBB) > 0).to_numpy()
other_only = db["superclasses"].apply(lambda s: len(s & set(CLASSES)) == 0).to_numpy()

table1_rows = {
    "All included ECGs": np.ones(len(db), bool),
    "NORM": Y[:, 0] == 1,
    "MI": Y[:, 1] == 1,
    "  Inferior MI (IMI)": is_imi,
    "  Anterior MI (AMI)": is_ami,
    "  Both IMI and AMI": is_imi & is_ami,
    "CD": Y[:, 2] == 1,
    "  Bundle branch block": is_bbb,
    "STTC or HYP only (negative for all)": other_only,
}
table1 = pd.DataFrame({name: [m[tr].sum(), m[va].sum(), m[te].sum(), m.sum()]
                       for name, m in table1_rows.items()},
                      index=["Train (folds 1-8)", "Validation (fold 9)", "Test (fold 10)", "Total"]).T
table1.to_csv(RESULTS / "table1_counts.csv")
table1

## 3. Signals: load, filter, standardize (Methods 2.2)
Reading all 21,388 ECGs takes a few minutes the first time; the raw array is then cached in `data/` (about 1 GB).

In [ ]:
RAW_CACHE = DATA / "signals_100hz_raw.npy"
if RAW_CACHE.exists():
    X_raw = np.load(RAW_CACHE)
else:
    names = wfdb.rdsamp(str(PTBXL_DIR / db["filename_lr"].iloc[0]))[1]["sig_name"]
    assert [n.upper() for n in names] == LEADS, f"Unexpected lead order: {names}"
    X_raw = np.empty((len(db), 1000, 12), dtype=np.float32)
    for i, f in enumerate(tqdm(db["filename_lr"], desc="Reading ECGs")):
        X_raw[i] = wfdb.rdsamp(str(PTBXL_DIR / f))[0]
    np.save(RAW_CACHE, X_raw)
print("Raw signals:", X_raw.shape, "| NaN values:", int(np.isnan(X_raw).sum()))

In [ ]:
# 4th-order Butterworth band-pass, 0.5-40 Hz, zero-phase (filtfilt), applied per lead
b, a = butter(4, [0.5, 40], btype="bandpass", fs=100)
X = np.empty((len(db), 12, 1000), dtype=np.float32)          # channels x time for PyTorch
for s in range(0, len(db), 2000):
    X[s:s + 2000] = filtfilt(b, a, X_raw[s:s + 2000], axis=1).transpose(0, 2, 1)
del X_raw

# Standardize each lead with the mean and SD of the TRAINING ECGs only
lead_mean = X[tr].mean(axis=(0, 2), keepdims=True)
lead_std = X[tr].std(axis=(0, 2), keepdims=True)
X -= lead_mean
X /= lead_std
signal_gb = X.nbytes / 1e9
print(f"CNN input: {X.shape} float32, {signal_gb:.2f} GB in memory")

## 4. Features for the random forest (Methods 2.3)
GE 12SL measurements from PTB-XL+. Column names follow `<Measure>_<Lead>`; a column whose suffix is a lead name is a per-lead feature, and everything else is global. **Check the printout:** the global features should be measurements only (no age, sex, or diagnosis fields).

In [ ]:
feat = pd.read_csv(PLUS_DIR / "12sl_features.csv")
key = "ecg_id" if "ecg_id" in feat.columns else feat.columns[0]
feat = feat.set_index(key).select_dtypes("number")
n_no_features = int((~db.index.isin(feat.index)).sum())
feat = feat.reindex(db.index)            # ECGs missing from 12SL become all-NaN rows (imputed below)
n_features_total = feat.shape[1]


def lead_of(col):
    suffix = col.rsplit("_", 1)[-1].upper() if "_" in col else ""
    return suffix if suffix in LEADS else "GLOBAL"


missing_rate = feat.iloc[tr].isna().mean()
keep = missing_rate[missing_rate <= 0.20].index
medians = feat.iloc[tr][keep].median()
F = feat[keep].fillna(medians).to_numpy(np.float32)
feature_names = list(keep)
feature_lead = np.array([lead_of(c) for c in feature_names])

print(f"{n_features_total} numeric 12SL features; dropped {n_features_total - len(keep)} "
      f"missing in >20% of training ECGs; kept {len(keep)}.")
print(f"{n_no_features} included ECGs had no 12SL row (all features imputed).")
print("\nFeatures per lead:")
print(pd.Series(feature_lead).value_counts().reindex(LEADS + ["GLOBAL"], fill_value=0).to_string())
missing_leads = [l for l in LEADS if l not in feature_lead]
if missing_leads:
    print("\nNote: 12SL has no per-lead features for", missing_leads)
print("\nGlobal features:", [c for c, l in zip(feature_names, feature_lead) if l == "GLOBAL"])

## 5. Random forest (Methods 2.4)
One balanced random forest per class, tuned on fold 9 over a 36-setting grid. The selected settings are saved to `results/rf_selected_params.json`, so later runs refit only the three final forests.

In [ ]:
GRID = list(itertools.product([200, 500], [10, 20, None], [1, 5, 20], ["sqrt", 0.3]))
PARAMS_FILE = RESULTS / "rf_selected_params.json"


def make_rf(p):
    return RandomForestClassifier(n_estimators=p["n_estimators"], max_depth=p["max_depth"],
                                  min_samples_leaf=p["min_samples_leaf"], max_features=p["max_features"],
                                  class_weight="balanced", random_state=SEED, n_jobs=-1)


rf_models = {}
if PARAMS_FILE.exists():
    best_params = json.loads(PARAMS_FILE.read_text())
else:
    best_params, log = {}, []
    for k, cls in enumerate(CLASSES):
        best_auc = -np.inf
        for n, depth, leaf, mf in tqdm(GRID, desc=f"Tuning {cls}"):
            p = {"n_estimators": n, "max_depth": depth, "min_samples_leaf": leaf, "max_features": mf}
            rf = make_rf(p).fit(F[tr], Y[tr, k])
            auc = roc_auc_score(Y[va, k], rf.predict_proba(F[va])[:, 1])
            log.append({"class": cls, **p, "val_auroc": auc})
            if auc > best_auc:
                best_auc, best_params[cls], rf_models[cls] = auc, {**p, "val_auroc": auc}, rf
            else:
                del rf
    pd.DataFrame(log).to_csv(RESULTS / "rf_grid_search.csv", index=False)
    PARAMS_FILE.write_text(json.dumps(best_params, indent=2))

for k, cls in enumerate(CLASSES):
    if cls not in rf_models:
        rf_models[cls] = make_rf(best_params[cls]).fit(F[tr], Y[tr, k])

pd.DataFrame(best_params).T

In [ ]:
# Sensitivity check: median imputation vs. dropping incomplete training rows (validation AUROC)
complete = ~feat.iloc[tr][keep].isna().any(axis=1).to_numpy()
tr_complete = tr[complete]
rows = []
for k, cls in enumerate(CLASSES):
    rf_drop = make_rf(best_params[cls]).fit(F[tr_complete], Y[tr_complete, k])
    rows.append({"class": cls,
                 "val AUROC, median imputation": best_params[cls]["val_auroc"],
                 "val AUROC, drop incomplete rows": roc_auc_score(Y[va, k], rf_drop.predict_proba(F[va])[:, 1])})
    del rf_drop
sensitivity = pd.DataFrame(rows).set_index("class").round(4)
sensitivity.to_csv(RESULTS / "rf_missing_data_sensitivity.csv")
print(f"Dropping incomplete rows keeps {len(tr_complete):,} of {len(tr):,} training ECGs.")
sensitivity

## 6. 1D CNN (Methods 2.5)
Four convolution blocks (32→64→128→128 channels, kernel 7), global average pooling, dropout 0.3, and 3 sigmoid outputs. Trained with weighted binary cross-entropy, Adam, and early stopping on validation macro-AUROC. Each seed's best weights are saved in `models/`, so reruns skip training.

In [ ]:
class ECGCNN(nn.Module):
    def __init__(self, n_leads=12, n_out=3, channels=(32, 64, 128, 128), kernel=7, dropout=0.3):
        super().__init__()
        layers, c_in = [], n_leads
        for c in channels:
            layers += [nn.Conv1d(c_in, c, kernel, padding=kernel // 2), nn.BatchNorm1d(c),
                       nn.ReLU(), nn.MaxPool1d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(c_in, n_out)

    def forward(self, x):                        # x: batch x 12 x 1000
        x = self.features(x).mean(dim=2)         # global average pooling over time
        return self.out(self.dropout(x))         # logits


n_cnn_params = sum(p.numel() for p in ECGCNN().parameters() if p.requires_grad)
print(f"Trainable parameters: {n_cnn_params:,}")


def predict(model, data, batch=512, logits=False):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(data), batch):
            z = model(torch.from_numpy(data[i:i + batch]).to(device))
            out.append((z if logits else torch.sigmoid(z)).cpu().numpy())
    return np.concatenate(out)


def macro_auroc(y, p):
    return float(np.mean([roc_auc_score(y[:, k], p[:, k]) for k in range(y.shape[1])]))


def train_cnn(seed, max_epochs=50, patience=10):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = ECGCNN().to(device)
    pos = Y[tr].sum(0)
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor((len(tr) - pos) / pos, dtype=torch.float32, device=device))
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=3)
    loader = DataLoader(TensorDataset(torch.from_numpy(X[tr]), torch.from_numpy(Y[tr])),
                        batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(seed))
    X_va, Y_va_t = X[va], torch.from_numpy(Y[va]).to(device)
    history, best_auc, best_state, wait = [], -np.inf, None, 0
    for epoch in range(1, max_epochs + 1):
        model.train()
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
        z_va = predict(model, X_va, logits=True)
        val_loss = loss_fn(torch.from_numpy(z_va).to(device), Y_va_t).item()
        val_auc = macro_auroc(Y[va], 1 / (1 + np.exp(-z_va)))
        history.append({"epoch": epoch, "train_loss": total / len(tr), "val_loss": val_loss,
                        "val_macro_auroc": val_auc, "lr": opt.param_groups[0]["lr"]})
        print(f"seed {seed} | epoch {epoch:2d} | train loss {total / len(tr):.4f} | "
              f"val loss {val_loss:.4f} | val macro-AUROC {val_auc:.4f}")
        sched.step(val_auc)
        if val_auc > best_auc:
            best_auc, wait = val_auc, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}")
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
cnn_models, histories, best_epochs = {}, {}, {}
for seed in CNN_SEEDS:
    weights, hist_file = MODELS / f"cnn_seed{seed}.pt", RESULTS / f"cnn_history_seed{seed}.csv"
    if weights.exists() and hist_file.exists():
        model = ECGCNN().to(device)
        model.load_state_dict(torch.load(weights, map_location=device))
        hist = pd.read_csv(hist_file)
    else:
        t0 = time.time()
        model, hist = train_cnn(seed)
        torch.save(model.state_dict(), weights)
        hist.to_csv(hist_file, index=False)
        print(f"seed {seed} trained in {(time.time() - t0) / 60:.1f} min")
    cnn_models[seed], histories[seed] = model, hist
    best_epochs[seed] = int(hist.loc[hist["val_macro_auroc"].idxmax(), "epoch"])
    print(f"seed {seed}: best epoch {best_epochs[seed]}, "
          f"validation macro-AUROC {hist['val_macro_auroc'].max():.4f}")
cnn = cnn_models[SEED]        # primary model used for RQ1 and RQ2

In [ ]:
# Fig. 2: training curves for the primary CNN
h, best = histories[SEED], best_epochs[SEED]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
ax1.plot(h["epoch"], h["train_loss"], label="Training", color="#1f4e79")
ax1.plot(h["epoch"], h["val_loss"], label="Validation", color="#e07b39")
ax1.set(xlabel="Epoch", ylabel="Weighted BCE loss", title="Loss")
ax1.legend(frameon=False)
ax2.plot(h["epoch"], h["val_macro_auroc"], color="#1f4e79")
ax2.set(xlabel="Epoch", ylabel="Macro-AUROC", title="Validation macro-AUROC (fold 9)")
for ax in (ax1, ax2):
    ax.axvline(best, color="gray", ls="--", lw=1)
ax2.annotate(f"best epoch {best}", (best, h["val_macro_auroc"].max()),
             xytext=(5, -15), textcoords="offset points", color="gray", fontsize=9)
fig.tight_layout()
fig.savefig(FIGS / "fig2_training_curves.png", bbox_inches="tight")
plt.show()

## 7. STOP: fix the expected lead groups, then commit

This cell writes `results/expected_lead_groups.json`. **Before running Section 8, commit and push this file** (VS Code Source Control → message "Fix expected lead groups before test evaluation" → Commit → Sync). The commit date is the evidence for Methods 2.7 that the criteria were fixed before the test set was used. Copy the commit hash into the Methods.

In [ ]:
EXPECTED = {"Inferior MI": ["Inferior"],
            "Anterior MI": ["Anterior"],
            "Bundle branch block": ["Anterior", "Lateral"]}
exp_file = RESULTS / "expected_lead_groups.json"
if exp_file.exists():
    assert json.loads(exp_file.read_text())["expected"] == EXPECTED, "Expected groups were changed after being fixed!"
    print("Expected groups already fixed:", exp_file)
else:
    exp_file.write_text(json.dumps({"lead_groups": GROUPS, "expected": EXPECTED,
                                    "written": time.strftime("%Y-%m-%d %H:%M")}, indent=2))
    print("Wrote", exp_file, "-> commit and push it now, then continue.")

## 8. RQ1: test-set performance (Methods 2.6)
Per-class and macro AUROC on fold 10, 95% CIs from 2,000 patient-level bootstrap resamples, and a paired bootstrap of the CNN − forest difference (Bonferroni α = 0.05/4 = 0.0125).

In [ ]:
y_te = Y[te]
p_rf = np.column_stack([rf_models[c].predict_proba(F[te])[:, 1] for c in CLASSES])
p_cnn_seeds = {s: predict(m, X[te]) for s, m in cnn_models.items()}
p_cnn = p_cnn_seeds[SEED]

# Patient-level bootstrap: resample test patients, keep all of each patient's ECGs
rng = np.random.default_rng(SEED)
pat_ids, pat_index = np.unique(patient[te], return_inverse=True)
rows_of_patient = pd.Series(np.arange(len(te))).groupby(pat_index).apply(np.array).tolist()
boot = [np.concatenate([rows_of_patient[i] for i in rng.integers(0, len(pat_ids), len(pat_ids))])
        for _ in range(N_BOOT)]
print(f"{N_BOOT} resamples of {len(pat_ids):,} test patients ({len(te):,} ECGs)")

In [ ]:
LABELS = CLASSES + ["Macro"]
ALPHA = 0.05 / len(LABELS)


def aurocs(y, p):
    per = [roc_auc_score(y[:, k], p[:, k]) for k in range(len(CLASSES))]
    return per + [float(np.mean(per))]


def ci(a):
    return np.nanpercentile(a, [2.5, 97.5], axis=0)


def p_two_sided(d):
    d = d[~np.isnan(d)]
    return min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()))


def fmt_p(p):
    return f"< {1 / N_BOOT:.4f}" if p == 0 else f"{p:.4f}"


point_rf, point_cnn = np.array(aurocs(y_te, p_rf)), np.array(aurocs(y_te, p_cnn))
boot_rf = np.array([aurocs(y_te[b], p_rf[b]) for b in tqdm(boot, desc="Bootstrap forest")])
boot_cnn = np.array([aurocs(y_te[b], p_cnn[b]) for b in tqdm(boot, desc="Bootstrap CNN")])
boot_diff = boot_cnn - boot_rf
ci_rf, ci_cnn, ci_diff = ci(boot_rf), ci(boot_cnn), ci(boot_diff)
p_vals = [p_two_sided(boot_diff[:, j]) for j in range(len(LABELS))]

rq1 = pd.DataFrame({
    "forest_auroc": point_rf, "forest_ci_low": ci_rf[0], "forest_ci_high": ci_rf[1],
    "cnn_auroc": point_cnn, "cnn_ci_low": ci_cnn[0], "cnn_ci_high": ci_cnn[1],
    "diff_cnn_minus_forest": point_cnn - point_rf, "diff_ci_low": ci_diff[0], "diff_ci_high": ci_diff[1],
    "p_paired_bootstrap": p_vals, "significant_bonferroni": [p < ALPHA for p in p_vals]}, index=LABELS)
rq1.to_csv(RESULTS / "table2_rq1_auroc.csv")

table2 = pd.DataFrame({
    "Forest AUROC (95% CI)": [f"{a:.3f} ({l:.3f}–{h:.3f})" for a, l, h in zip(point_rf, *ci_rf)],
    "CNN AUROC (95% CI)": [f"{a:.3f} ({l:.3f}–{h:.3f})" for a, l, h in zip(point_cnn, *ci_cnn)],
    "Difference, CNN − forest (95% CI)": [f"{a:+.3f} ({l:+.3f} to {h:+.3f})"
                                          for a, l, h in zip(point_cnn - point_rf, *ci_diff)],
    "p": [fmt_p(p) for p in p_vals],
    f"Significant (α = {ALPHA:.4f})": ["Yes" if p < ALPHA else "No" for p in p_vals]}, index=LABELS)

seed_auc = pd.DataFrame({f"seed {s}": aurocs(y_te, p) for s, p in p_cnn_seeds.items()}, index=LABELS)
seed_auc["mean"], seed_auc["min"], seed_auc["max"] = (seed_auc.mean(1), seed_auc.min(1), seed_auc.max(1))
seed_auc.round(4).to_csv(RESULTS / "cnn_seed_stability.csv")
print("CNN test AUROC across seeds:")
display(seed_auc.round(4))
table2

In [ ]:
# Fig. 3: AUROC difference (CNN - forest) with 95% CI
fig, ax = plt.subplots(figsize=(6, 3.6))
x = np.arange(len(LABELS))
d = point_cnn - point_rf
ax.errorbar(x, d, yerr=[d - ci_diff[0], ci_diff[1] - d], fmt="o", color="#1f4e79",
            capsize=5, markersize=7, lw=1.5)
ax.axhline(0, color="gray", lw=1, ls="--")
for xi, di, p in zip(x, d, p_vals):
    ax.annotate(f"{di:+.3f}" + ("*" if p < ALPHA else ""), (xi, di), xytext=(10, -4),
                textcoords="offset points", fontsize=9)
ax.set_xticks(x, ["NORM", "MI", "CD", "Macro"])
ax.set_ylabel("AUROC difference\n(CNN − forest)")
ax.set_title("Test-set AUROC gap with 95% bootstrap CI\n(* significant after Bonferroni correction)", fontsize=10)
fig.tight_layout()
fig.savefig(FIGS / "fig3_auroc_difference.png", bbox_inches="tight")
plt.show()

## 9. RQ2: which lead groups each model relies on (Methods 2.7)
Each condition is tested on its own test subset: inferior MI (IMI only) and anterior MI (AMI only) against ECGs with no MI, and bundle branch block against ECGs with no CD.

In [ ]:
CONDITIONS = {
    "Inferior MI": {"pos": is_imi[te] & ~is_ami[te], "neg": Y[te, 1] == 0, "k": 1},
    "Anterior MI": {"pos": is_ami[te] & ~is_imi[te], "neg": Y[te, 1] == 0, "k": 1},
    "Bundle branch block": {"pos": is_bbb[te], "neg": Y[te, 2] == 0, "k": 2},
}
GNAMES = list(GROUPS)


def value(df, condition, group, column):
    """One value from a long-format results table."""
    return df.loc[(df["condition"] == condition) & (df["group"] == group), column].item()


for name, c in CONDITIONS.items():
    c["subset"] = c["pos"] | c["neg"]
    print(f"{name}: {c['pos'].sum()} positives, {c['neg'].sum()} negatives "
          f"(model output: {CLASSES[c['k']]}; expected group: {' or '.join(EXPECTED[name])})")

In [ ]:
# CNN occlusion: set one lead group to zero (= training mean after standardization) on every test ECG
lead_pos = {l: i for i, l in enumerate(LEADS)}
X_te = X[te]
p_occluded = {}
for g, leads in GROUPS.items():
    X_occ = X_te.copy()
    X_occ[:, [lead_pos[l] for l in leads], :] = 0.0
    p_occluded[g] = predict(cnn, X_occ)
del X_occ


def subset_auroc(cond, probs, idx=None):
    idx = np.arange(len(te)) if idx is None else idx
    m = cond["subset"][idx]
    y = cond["pos"][idx][m]
    if y.min() == y.max():
        return np.nan
    return roc_auc_score(y, probs[idx][m, cond["k"]])


occl_rows, occl_boot = [], {}
for name, c in CONDITIONS.items():
    intact = subset_auroc(c, p_cnn)
    drops = np.array([intact - subset_auroc(c, p_occluded[g]) for g in GNAMES])
    bd = np.array([[subset_auroc(c, p_cnn, b) - subset_auroc(c, p_occluded[g], b) for g in GNAMES]
                   for b in tqdm(boot, desc=name)])
    occl_boot[name] = bd
    lo, hi = ci(bd)
    for j, g in enumerate(GNAMES):
        occl_rows.append({"condition": name, "group": g, "intact_auroc": intact, "auroc_drop": drops[j],
                          "ci_low": lo[j], "ci_high": hi[j], "expected": g in EXPECTED[name]})
occlusion = pd.DataFrame(occl_rows)
occlusion.to_csv(RESULTS / "rq2_cnn_occlusion.csv", index=False)

# Paired test: is the drop for the expected group larger than for each other group?
paired_rows = []
for name, bd in occl_boot.items():
    for e in EXPECTED[name]:
        for o in GNAMES:
            if o in EXPECTED[name]:
                continue
            d = bd[:, GNAMES.index(e)] - bd[:, GNAMES.index(o)]
            point = value(occlusion, name, e, "auroc_drop") - value(occlusion, name, o, "auroc_drop")
            lo, hi = np.nanpercentile(d, [2.5, 97.5])
            paired_rows.append({"condition": name, "expected_group": e, "other_group": o,
                                "drop_difference": point, "ci_low": lo, "ci_high": hi, "p": p_two_sided(d)})
occl_paired = pd.DataFrame(paired_rows)
occl_paired.to_csv(RESULTS / "rq2_cnn_occlusion_paired.csv", index=False)
display(occlusion.round(4))
occl_paired.round(4)

In [ ]:
# Random forest SHAP, summed by lead group (TreeExplainer can take several minutes per condition)
in_group = {g: np.isin(feature_lead, leads) for g, leads in GROUPS.items()}
in_group["Global"] = feature_lead == "GLOBAL"
PARTS = GNAMES + ["Global"]
leads_present = {g: max(1, len(set(feature_lead) & set(leads))) for g, leads in GROUPS.items()}
print("Leads with 12SL features in each group:", leads_present)


def abs_shap_positive(model, data):
    sv = shap.TreeExplainer(model).shap_values(data, check_additivity=False)
    if isinstance(sv, list):
        sv = sv[1]
    elif sv.ndim == 3:
        sv = sv[:, :, 1]
    return np.abs(sv)


shap_rows, shap_first = [], {}
for name, c in CONDITIONS.items():
    pos_idx = np.where(c["pos"])[0]
    t0 = time.time()
    a = abs_shap_positive(rf_models[CLASSES[c["k"]]], F[te][pos_idx])
    per_ecg = np.column_stack([a[:, in_group[p]].sum(1) for p in PARTS])    # positives x 4
    mean = per_ecg.mean(0)
    share3 = mean[:3] / mean[:3].sum()          # anatomical groups only (used for ranking)
    share4 = mean / mean.sum()                  # including global (Fig. 5)
    per_lead = mean[:3] / np.array([leads_present[g] for g in GNAMES])
    # Bootstrap stability: which group ranks first in each patient resample
    where = np.full(len(te), -1)
    where[pos_idx] = np.arange(len(pos_idx))
    firsts = []
    for b in boot:
        r = where[b]
        r = r[r >= 0]
        if len(r):
            firsts.append(GNAMES[int(np.argmax(per_ecg[r, :3].mean(0)))])
    shap_first[name] = pd.Series(firsts).value_counts(normalize=True)
    for j, p in enumerate(PARTS):
        shap_rows.append({"condition": name, "group": p, "mean_abs_shap_sum": mean[j],
                          "share_of_anatomical": share3[j] if j < 3 else np.nan,
                          "share_incl_global": share4[j],
                          "per_lead_importance": per_lead[j] if j < 3 else np.nan,
                          "expected": p in EXPECTED[name]})
    print(f"{name}: SHAP on {len(pos_idx)} ECGs in {(time.time() - t0) / 60:.1f} min")
shap_groups = pd.DataFrame(shap_rows)
shap_groups.to_csv(RESULTS / "rq2_forest_shap_groups.csv", index=False)
shap_groups.round(4)

In [ ]:
# Table 3: does each model's most important group match the expected group?
rows = []
for name in CONDITIONS:
    exp = EXPECTED[name]
    occ = occlusion[occlusion["condition"] == name].set_index("group")["auroc_drop"]
    bd = occl_boot[name]
    valid = ~np.isnan(bd).any(1)
    cnn_first = pd.Series(np.array(GNAMES)[np.argmax(bd[valid], 1)]).value_counts(normalize=True)
    shp = shap_groups[(shap_groups["condition"] == name) & shap_groups["group"].isin(GNAMES)].set_index("group")
    for model_name, scores, firsts in [("Random forest (SHAP share)", shp["share_of_anatomical"], shap_first[name]),
                                       ("CNN (occlusion AUROC drop)", occ, cnn_first)]:
        ranking = scores.sort_values(ascending=False)
        rows.append({"condition": name, "model": model_name, "expected": " or ".join(exp),
                     "ranking": " > ".join(ranking.index),
                     "top_group": ranking.index[0], "consistent": ranking.index[0] in exp,
                     "bootstrap_fraction_expected_first": float(sum(firsts.get(e, 0) for e in exp))})
table3 = pd.DataFrame(rows)
table3.to_csv(RESULTS / "table3_rq2_consistency.csv", index=False)
table3

In [ ]:
# Fig. 4: forest SHAP share vs. CNN occlusion drop, same layout, expected group outlined
cond_names = list(CONDITIONS)
M_rf = np.array([[value(shap_groups, c, g, "share_of_anatomical") for g in GNAMES] for c in cond_names])
M_cnn = np.array([[value(occlusion, c, g, "auroc_drop") for g in GNAMES] for c in cond_names])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
panels = [(M_rf, "Random forest\nSHAP share of importance", "{:.0%}"),
          (M_cnn, "CNN\nAUROC drop when group is hidden", "{:.3f}")]
for ax, (M, title, f) in zip(axes, panels):
    im = ax.imshow(M, cmap="Blues", aspect="auto", vmin=min(0, M.min()))
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f.format(M[i, j]), ha="center", va="center",
                    color="white" if M[i, j] > M.max() * 0.6 else "black", fontsize=10)
            if GNAMES[j] in EXPECTED[cond_names[i]]:
                ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="#e07b39", lw=3))
    ax.set_xticks(range(3), GNAMES)
    ax.set_yticks(range(3), cond_names)
    ax.set_title(title, fontsize=10)
    ax.spines[:].set_visible(False)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Lead-group importance by condition (orange box = group expected from clinical criteria)", fontsize=11)
fig.tight_layout()
fig.savefig(FIGS / "fig4_lead_group_importance.png", bbox_inches="tight")
plt.show()

In [ ]:
# Fig. 5: forest SHAP shares including global features
colors = {"Inferior": "#1f4e79", "Anterior": "#5b9bd5", "Lateral": "#9dc3e6", "Global": "#bfbfbf"}
fig, ax = plt.subplots(figsize=(7, 3.4))
left = np.zeros(len(cond_names))
for p in PARTS:
    vals = np.array([value(shap_groups, c, p, "share_incl_global") for c in cond_names])
    ax.barh(cond_names, vals, left=left, color=colors[p], label=p)
    for i, v in enumerate(vals):
        if v > 0.06:
            ax.text(left[i] + v / 2, i, f"{v:.0%}", ha="center", va="center",
                    color="white" if p in ("Inferior", "Anterior") else "black", fontsize=9)
    left += vals
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.xaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlabel("Share of total mean |SHAP|")
ax.legend(ncol=4, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.18))
fig.tight_layout()
fig.savefig(FIGS / "fig5_forest_shap_shares.png", bbox_inches="tight")
plt.show()

## 10. Summary for the paper
Saves `results/results_summary.json` with every number the Methods and Results need. **Send this file and the `figures/` folder back to Claude** to write the Results section.

In [ ]:
versions = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "scipy": scipy.__version__, "scikit-learn": sklearn.__version__, "shap": shap.__version__,
            "torch": torch.__version__, "wfdb": wfdb.__version__, "matplotlib": matplotlib.__version__}
summary = {
    "methods_fill_ins": {
        "signal_memory_gb": round(signal_gb, 2),
        "n_12sl_features_total": int(n_features_total),
        "n_12sl_features_kept": int(len(keep)),
        "n_ecgs_without_12sl_row": n_no_features,
        "n_training_ecgs_complete_rows": int(len(tr_complete)),
        "features_per_lead": pd.Series(feature_lead).value_counts().to_dict(),
        "rf_selected_params": best_params,
        "cnn_trainable_params": int(n_cnn_params),
        "cnn_best_epochs": best_epochs,
        "device": str(device), "machine": platform.platform(),
        "versions": versions,
    },
    "table1": table1.to_dict(),
    "rf_missing_data_sensitivity": sensitivity.to_dict(),
    "rq1_table2": rq1.round(4).to_dict(orient="index"),
    "cnn_seed_stability": seed_auc.round(4).to_dict(orient="index"),
    "rq2_cnn_occlusion": occlusion.round(4).to_dict(orient="records"),
    "rq2_cnn_occlusion_paired": occl_paired.round(4).to_dict(orient="records"),
    "rq2_forest_shap": shap_groups.round(4).to_dict(orient="records"),
    "rq2_table3": table3.round(4).to_dict(orient="records"),
}
(RESULTS / "results_summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("Saved", RESULTS / "results_summary.json")
print(json.dumps(versions, indent=2))